In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns

import glob
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.graph_objs as go

# from Bio import SeqIO

import gzip

import h5py
import scanpy as sc
import scipy
import mira
import torch
import anndata as ad

In [ ]:
#mira.__version__,torch.__version__

In [ ]:
# import plotly.plotly as py
# import plotly.figure_factory as FF
# import cufflinks as cf

# cf.set_config_file(offline=True, world_readable=True, theme='white')
# cf.set_config_file(offline=False, world_readable=True, theme='ggplot')
# py.sign_in('Jingyu', 'zmgxyo3uk9')

In [ ]:
import logging
import warnings
#mira.utils.pretty_sderr()

In [ ]:
%config Completer.use_jedi = False

In [ ]:
%matplotlib inline

In [ ]:
torch.cuda.is_available()

In [ ]:
#for i in range(torch.cuda.device_count()):
#    print(torch.cuda.get_device_properties(i).name)

In [ ]:
#torch.cuda.get_device_properties(0)

In [ ]:
cd /ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/

# preprocess

In [ ]:
adata=sc.read_h5ad("/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/h5_files/merged.h5ad")

In [ ]:
adata.var['gene_ids'].head(200)

In [ ]:
adata.obs_names_make_unique()

In [ ]:
rawdata = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.layers['counts'] = rawdata

In [ ]:
sc.pp.highly_variable_genes(adata, min_disp = 0.2)

In [ ]:
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# sc.pp.highly_variable_genes(adata, min_disp = 0.2)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
adata

In [ ]:
adata.var['highly_variable'].value_counts()

In [ ]:
adata.obs['group'].unique()

In [ ]:


# Assuming `adata` is your AnnData object
adata.obs['batch'] = adata.obs['group'].apply(
    lambda x: 'donA' if 'donA' in x else ('donB' if 'donB' in x else x))



In [ ]:
adata.obs['batch'].unique()

# MIRA

In [ ]:
model = mira.topics.make_model(
    adata.n_obs, adata.n_vars, # helps MIRA choose reasonable values for some hyperparameters which are not tuned.
    feature_type = 'expression',
    highly_variable_key='highly_variable',
    counts_layer='counts',
    categorical_covariates='batch'
)

In [ ]:
model.get_learning_rate_bounds(adata)

In [ ]:
model.set_learning_rates(1e-3, 0.25)
model.plot_learning_rate_bounds(figsize=(7,3))

# Hyperparameter Optimization

# Method 1: Gradient based

In [ ]:
#takes a long time (30min)
topic_contributions = mira.topics.gradient_tune(model, adata)

In [ ]:
with open('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/topic_contributions.txt', 'w') as file:
    file.write('\n'.join(str(topic) for topic in topic_contributions))

In [ ]:
NUM_TOPICS = 30

mira.pl.plot_topic_contributions(topic_contributions, NUM_TOPICS)

In [ ]:
#takes ~5-10min
model = model.set_params(num_topics = NUM_TOPICS).fit(adata)

In [ ]:
model.save('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/models/gradient_model_1.pth')

In [ ]:
#reload model if needed
model = mira.topic_model.load_model('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/models/gradient_model_1.pth')



In [ ]:
model

# (Optional) Method 2: Bayesian Optimization

In [ ]:
tuner = mira.topics.BayesianTuner(
        model = model,
        n_jobs=2,
        save_name = '/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/models/expression_model_tuner',
        #### IMPORTANT
        min_topics = 27, max_topics = 33, # tailor for your dataset!!!! (+/- 10 from the above topic number)
        #### See "Notes on min_topics, max_topics" above
        #storage = mira.topics.Redis() # if using REDIS backend for more (>5) processes
)

In [ ]:
tuner

In [ ]:
#takes the longest (~2-4hrs)
tuner.fit(adata)

In [ ]:
ax = tuner.plot_intermediate_values(palette='Spectral_r',
                                   log_hue=True, figsize=(7,3))
# ax.set(ylim = (7e2, 7.7e2))

In [ ]:
tuner.plot_pareto_front(include_pruned_trials=False, label_pareto_front=True,
                       figsize = (5,5))

In [ ]:
model = tuner.fetch_best_weights()

In [ ]:
model.save('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/models/bayesian_model_1.pth')

In [ ]:
#uses topic model to predict for each cell
model.predict(adata)

In [ ]:
adata.write('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/h5_files/merged_topic_modeling.h5ad')

# scanpy workflow

In [ ]:
adata = sc.read_h5ad('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/h5_files/merged_topic_modeling.h5ad')

In [ ]:
model = mira.topic_model.load_model('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/models/bayesian_model_1.pth')

In [ ]:
model.get_umap_features(adata, box_cox=0.25)

In [ ]:
sc.pp.neighbors(adata, use_rep = 'X_umap_features', metric = 'manhattan', n_neighbors = 15)

In [ ]:
sc.tl.umap(adata, min_dist=0.5, negative_sample_rate=1, spread=1)


# Rotate UMAP coordinates by -180 degrees
angle_rad = np.deg2rad(-90)
rotation_matrix = np.array([
    [np.cos(angle_rad), -np.sin(angle_rad)],
    [np.sin(angle_rad),  np.cos(angle_rad)]
])

adata.obsm['X_umap'] = adata.obsm['X_umap'].dot(rotation_matrix)


ax = sc.pl.umap(adata, frameon=False, color = 'group', s=15, show = False)

In [ ]:
ax = sc.pl.umap(adata, size=20, show=False)
sc.pl.umap(
    adata[adata.obs.group == "donA_bcl6_ko"],
    size=20,
    color="group",
    ax=ax,
    show=False,
    frameon=False,
    title=''
)

In [ ]:
ax = sc.pl.umap(adata, size=20, show=False)
sc.pl.umap(
    adata[adata.obs.group == "donA_ntc"],
    size=20,
    color="group",
    ax=ax,
    show=False,
    frameon=False
)

In [ ]:
ax = sc.pl.umap(adata, size=20, show=False)
sc.pl.umap(
    adata[adata.obs.group == "donB_bcl6_ko"],
    size=20,
    color="group",
    ax=ax,
    show=False,
    frameon=False
)

In [ ]:
ax = sc.pl.umap(adata, size=20, show=False)
sc.pl.umap(
    adata[adata.obs.group == "donB_ntc"],
    size=20,
    color="group",
    ax=ax,
    show=False,
    frameon=False
)

In [ ]:
list_g1s=['CDCA7','MCM4','MCM2','RFC2','UNG','MCM6','RRM1','SLBP','PCNA','ATAD2','TIPIN','MCM5','UHRF1','RPA2','DTL','PRIM1','FEN1','HELLS','GMNN','POLD3','NASP','CHAF1B','GINS2','POLA1','MSH2','CASP8AP2','CDC6','UBR7','CCNE2','WDR76','TYMS','CDC45','CLSPN','RRM2','DSCC1','RAD51','USP1','EXO1','BLM','RAD51AP1','CENPU','E2F8','BRIP1']
list_g2m=['CBX5','AURKB','CKS1B','CKS2','JPT1','HMGB2','ANP32E','LBR','TMPO','TOP2A','TACC3','TUBB4B','NCAPD2','RANGAP1','CDK1','SMC4','KIF20B','CDCA8','CKAP2','NDC80','DLGAP5','HJURP','CKAP5','BUB1','CKAP2L','ECT2','KIF11','BIRC5','CDCA2','NUF2','CDCA3','NUSAP1','TTK','AURKA','MKI67','PIMREG','CCNB2','TPX2','ANLN','KIF2C','CENPE','GTSE1','KIF23','CDC20','UBE2C','CENPF','CENPA','HMMR','CTCF','PSRC1','CDC25C','NEK2','GAS2L3','G2E3']
list_myc_up=['NARS','NUFIP1','NUP43','SLC19A1','NCL','EXOSC7','SURF2','PSMG1','METTL1','AEN','1110004E09RIK','LYAR','PSME3','TSR1','SHMT1','BCAT1','CCND2','NDUFAF4','SLC40A1','HEATR3','AIMP2','TTC27','FAM136A','VCAM1','CCND2','NOP56','C1QA','MKI67IP','AIMP2','PLSCR1','RRP1B','MFSD2A','CIRH1A','C1QB','MRTO4','CCND2','WDR75','HSP90AA1','GCSH','FAM195A','TIMM10','FABP5','SRM','C1QC','2610528E23RIK','LOC100047619','CCND2','SAA3','UTP15','ATIC','SHMT1','THYN1','2700023E23RIK','DDX18','IL1R2','TIMM9','CSDA','RCL1','GRWD1','NOLC1','GM12816','RPF2','1810029B16RIK','GSTT1','TSR1','TOMM5','CCT6A','NHP2','CHCHD4','MOGS','IRF4','FSCN1','GM7901','SLC19A1','PLSCR1','PA2G4','HSPD1','LOC100047009','EBNA1BP2','PHB','MATR3','GAR1','RCC1','NOP58','SLC19A1','GNL3','TSEN2','NDUFAF4','EMR4','GSTT2','SLPI','ABCG2','NOP16','PPP1R14B','LAP3','PRDX6','SHMT2','PPP1R14B','GM14005','NOLC1','LPL','MATR3','WDR12','BATF','C1QBP','---','1110004E09RIK','CCND2','UCK2','SPIRE1','LOC100048307','HTRA2','DDX18','PSMD7','TUBA4A','2610201A13RIK','CCND2','NOC4L','RPP40','ASNS','LYZ1','RANGRF','SUB1','ENDOG','DCTD','BSN','RSL1D1','NT5DC2','NHP2','2010204K13RIK','FAM185A','PECR','UCK2','SOCS3','ACY1','ZMYND19','SOCS3','PRMT1','SMYD2','FABP5','FBL','GRWD1','DDT','PNO1','CCND2','PA2G4','GRWD1','APRT','HK2','CCDC86','ECE2','APEX1','SOCS2','ZNHIT6','ATAD3A','HSPA9','TIMD2','PPA1','MIF','CTPS','APEX1','MRPS18B','IL10','METTL1','TUBA4A','NCL','NOP16','MRPS6','HSP90B1','TCFAP4','EIF2B3','SHMT1','RANBP1','ASNS','GSR','NOC4L','DCTPP1','LOC639633','2610019E17RIK','UCK2','ADM','TMEM158','CD70','POLR1B','MYC','C1QBP','PPAT','PYCR1','SPR']

In [ ]:
sc.tl.score_genes(adata, 
                   list_g1s, 
                   ctrl_size=300, 
                   gene_pool=None, 
                   n_bins=25, 
                   score_name='score_G1S', 
                   random_state=6, 
                   copy=False, 
                   use_raw=False)


sc.tl.score_genes(adata, 
                   list_g2m, 
                   ctrl_size=300, 
                   gene_pool=None, 
                   n_bins=25, 
                   score_name='score_G2M', 
                   random_state=6, 
                   copy=False, 
                   use_raw=False)


sc.tl.score_genes(adata, 
                   list_myc_up, 
                   ctrl_size=300, 
                   gene_pool=None, 
                   n_bins=25, 
                   score_name='score_myc_up', 
                   random_state=6, 
                   copy=False, 
                   use_raw=False)


In [ ]:
gene_set = ['IRF8', 'IRF4', 'MKI67', 'score_G1S', 'score_G2M']
sc.set_figure_params(scanpy=True, fontsize=20, dpi_save = 350)
sns.set(rc={'figure.figsize':(4,4)})

sc.pl.umap(adata, 
           color=gene_set,
           color_map='viridis',
           frameon = False,
           show = False,
           s = 20,
           ncols=5
          )

In [ ]:
gene_set = ['CXCR4', 'CXCR5', 'LTB', 'TNFRSF13C', 'GCSAM']
sc.set_figure_params(scanpy=True, fontsize=20, dpi_save = 350)
sns.set(rc={'figure.figsize':(4,4)})

sc.pl.umap(adata, 
           color=gene_set,
           color_map='viridis',
           frameon = False,
           show = False,
           s = 20,
           ncols=5
          )


In [ ]:
sc.tl.leiden(adata,resolution=0.3,random_state=6, flavor="igraph")

In [ ]:
sc.pl.umap(adata, 
           color='leiden',
           color_map='viridis',
           frameon = False,
           show = False,
           s = 20,
           ncols=5
          )


In [ ]:
adata.write("/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/h5_files/merged_post_mira.h5ad"

In [ ]:
df_temp=pd.read_excel('/ix/cigcore/peasena/gene_sets/human_tonsil_atlas_GC_vs_ActB_preGCB.csv')
list_gc_vs_actb_pregc= df_temp['gene_name - selection'].dropna().tolist()
list_gc_vs_actb_pregc